In [5]:
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, Dataset
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import numpy as np

In [12]:
# Hyperparameters
layers = [3, 64, 64, 64, 1]
lr = 1e-3
batch_size = 32
epochs = 50

In [13]:
def sdf_box(p, center, half_extents):
    """
    SDF for an axis-aligned box.

    Parameters
    ----------
    p : (N, 2) array of points to evaluate
    center : (2,) array-like — center of the box e.g. [0.0, 0.0]
    half_extents : (2,) array-like — half the width and height of the box e.g. [0.5, 0.5]
    """
    q = np.abs(p - center) - half_extents
    return np.linalg.norm(np.maximum(q, 0), axis=-1) + np.minimum(np.max(q, axis=-1), 0)


def sdf_t_block(p, bar_width, bar_height, stem_width, stem_height):
    """
    T-shape as union of two boxes.
    Origin is at the center of the full bounding box.

        ┌─────────────┐  ↑
        │   top bar   │  bar_height
        └────┬───┬────┘  ↓
             │ s │  ↑
             │ t │  stem_height
             │ e │
             │ m │  ↓
             └───┘
    """
    bar_center = np.array([0.0, stem_height / 2])
    stem_center = np.array([0.0, -bar_height / 2])

    overlap = 0.05
    bar = sdf_box(p, bar_center, np.array([bar_width / 2, bar_height / 2]))
    stem = sdf_box(
        p, 
        stem_center + np.array([0.0, overlap / 2]), 
        np.array([stem_width / 2, stem_height / 2 + overlap / 2])
    )

    return np.minimum(bar, stem)

In [14]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Plate parameters
size = 2.0
t_max = 5.0
alpha = 0.01

SDF = lambda coords: sdf_t_block(
    coords, bar_width=1.5, bar_height=0.385, stem_width=0.385, stem_height=1.0
)

Using device: cuda


In [16]:
!wget -O T_plate.csv https://raw.githubusercontent.com/brcktn/Inverse-Heat-PINN/main/training_data/T_plate.csv

class CSVDataset(Dataset):
    def __init__(self, csv_file):
        data = pd.read_csv(csv_file).values.astype(np.float32)
        self.X = torch.from_numpy(data[:, 1:4])  # coords (x, y, t)
        self.y = torch.from_numpy(data[:, 4:])   # temperature

    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = CSVDataset('T_plate.csv')
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# next(iter(dataloader))

--2026-04-15 03:22:14--  https://raw.githubusercontent.com/brcktn/Inverse-Heat-PINN/main/training_data/T_plate.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 615246 (601K) [text/plain]
Saving to: ‘T_plate.csv’

T_plate.csv         100%[===================>] 600.83K  --.-KB/s    in 0.06s   

2026-04-15 03:22:14 (9.68 MB/s) - ‘T_plate.csv’ saved [615246/615246]



In [17]:
class HeatPINN(nn.Module):
    def __init__(self, layers):
        """
        Parameters
        ----------
        layers : list of int
            A list of integers specifying the number of neurons in each layer
        """
        super(HeatPINN, self).__init__()
        self.layers = nn.ModuleList()
        for i in range(len(layers) - 1):
            self.layers.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers) - 2:
                self.layers.append(nn.SiLU())

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

In [20]:
def sample_points(num_points, t_max, SDF):
    """
    Sample points in space and time within the domain defined by the SDF.
    
    Parameters
    ----------
    num_points : int
        Total number of points to sample
    t_max : float
        Maximum time to sample
    SDF : function
        A function that takes (N, 2) coordinates and returns their SDF values
    """
    points = []
    while len(points) < num_points:
        x = np.random.uniform(-size/2, size/2)
        y = np.random.uniform(-size/2, size/2)
        t = np.random.uniform(0, t_max)
        if SDF(np.array([[x, y]])) < 0:  # Inside the domain
            points.append([x, y, t])
    return np.array(points)


def sample_boundary_points(num_points, t_max, SDF, eps=1e-4, steps=10):
    """
    Sample points precisely on the boundary of the domain defined by the SDF
    using gradient descent projection.
    
    Parameters
    ----------
    num_points : int
        Total number of points to sample
    t_max : float
        Maximum time to sample
    SDF : function
        A function that takes (N, 2) coordinates and returns their SDF values
    """
    # Start with random points in domain for spatial coords (x,y) and time (t)
    pts_xy = np.random.uniform(-size/2, size/2, (num_points * 10, 2))
    pts_t = np.random.uniform(0, t_max, (num_points * 10, 1))
    
    # Project onto surface via numerical gradient descent on the SDF
    # (Since the SDF is currently implemented with NumPy, we use numerical gradients)
    dx = 1e-5
    for _ in range(steps):
        f = SDF(pts_xy)
        
        grad_x = (SDF(pts_xy + np.array([dx, 0])) - SDF(pts_xy - np.array([dx, 0]))) / (2 * dx)
        grad_y = (SDF(pts_xy + np.array([0, dx])) - SDF(pts_xy - np.array([0, dx]))) / (2 * dx)
        grad = np.stack([grad_x, grad_y], axis=-1)
        
        grad_norm = np.linalg.norm(grad, axis=-1, keepdims=True)
        pts_xy -= np.expand_dims(f, -1) * grad / (grad_norm + 1e-8)
    
    # Keep points that converged precisely to the surface
    f = SDF(pts_xy)
    on_surface = np.abs(f) < eps
    
    pts_xy_surface = pts_xy[on_surface]
    pts_t_surface = pts_t[on_surface]
    
    points = np.concatenate([pts_xy_surface, pts_t_surface], axis=-1)
    return points[:num_points]

In [ ]:
def data_loss(model, points, temperatures):
    """
    MSE loss between model predictions and observed temperatures at given points.

    Parameters
    ----------
    model : nn.Module
        The PINN model
    points : (N, 3) tensor
        The (x, y, t) coordinates of the observed data points
    temperatures : (N, 1) tensor
        The observed temperatures at the given points

    Returns
    -------
    torch.Tensor
        The computed MSE loss
    """
    x_batch = points.to(device)
    y_batch = temperatures.to(device)
    pred = model(x_batch)
    return nn.MSELoss()(pred, y_batch)
